# Uyghur (uig) — Arabic Script NLP Pipeline

Uyghur is written in Perso-Arabic script (right-to-left). TurkicNLP provides Arabic-script tokenisation, Apertium FST morphology (Beta quality), full Stanza neural pipeline via the UDT treebank, and bidirectional Arabic↔Latin (ULY) transliteration.

In [ ]:
# Install TurkicNLP
# pip install turkicnlp          # core (tokenization, transliteration)
# pip install "turkicnlp[stanza]"  # adds POS, lemma, depparse, NER
# pip install "turkicnlp[nllb]"    # adds cross-lingual embeddings + translation
# pip install "turkicnlp[all]"     # all optional dependencies

In [ ]:
import turkicnlp
from turkicnlp import Pipeline

## 1. Download Models

In [ ]:
turkicnlp.download('uig')

## 2. Script Detection — Arabic Script

In [ ]:
from turkicnlp.scripts.detector import detect_script

# Uyghur in Perso-Arabic script
arab_text = "مەن مەكتەپكە بارىمەن."
print("Detected:", detect_script(arab_text))

## 3. Arabic ↔ Latin (ULY) Transliteration

In [ ]:
from turkicnlp.scripts import Script
from turkicnlp.scripts.detector import detect_script
from turkicnlp.scripts.transliterator import Transliterator

print("=" * 70)
print("UYGHUR COMPREHENSIVE TRANSLITERATION")
print("=" * 70)
print()

arab = "مەن مەكتەپكە بارىمەن."
print(f"Original (Perso-Arabic): {arab}")
print()

# Direction 1: Perso-Arabic → Turkic Common Alphabet (Latin/ULY)
print("1. Perso-Arabic → Turkic Common Alphabet (Latin/ULY):")
try:
    t1 = Transliterator("uig", source=Script.ARABIC, target=Script.COMMON_TURKIC)
    common = t1.transliterate(arab)
    print(f"   {common}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 2: Turkic Common → Perso-Arabic (reverse)
print("2. Turkic Common (Latin) → Perso-Arabic:")
try:
    t2 = Transliterator("uig", source=Script.COMMON_TURKIC, target=Script.ARABIC)
    back_to_arab = t2.transliterate(common if 'common' in locals() else "Men mektepke barimen.")
    print(f"   {back_to_arab}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 3: Perso-Arabic → Latin (explicit)
print("3. Perso-Arabic → Latin (explicit):")
try:
    t3 = Transliterator("uig", source=Script.ARABIC, target=Script.LATIN)
    latin = t3.transliterate(arab)
    print(f"   {latin}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 4: Latin → Perso-Arabic
print("4. Latin → Perso-Arabic:")
try:
    t4 = Transliterator("uig", source=Script.LATIN, target=Script.ARABIC)
    back_to_arab_explicit = t4.transliterate(latin if 'latin' in locals() else "Men mektepke barimen.")
    print(f"   {back_to_arab_explicit}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

# Direction 5: Perso-Arabic → Cyrillic (if supported)
print("5. Perso-Arabic → Cyrillic (if supported):")
try:
    t5 = Transliterator("uig", source=Script.ARABIC, target=Script.CYRILLIC)
    cyrillic = t5.transliterate(arab)
    print(f"   {cyrillic}")
except Exception as e:
    print(f"   ⚠ Not supported: {e}")
print()

print("=" * 70)
print("Note: ZWNJ (U+200C) characters preserve morpheme boundaries in Arabic script")
print("=" * 70)

## 4. Morphological Analysis (Apertium FST — Beta)

In [ ]:
nlp_morph = Pipeline(
    "uig",
    processors=["tokenize", "morph"],
    morph_backend="apertium",
    script="Arab",
)
doc = nlp_morph("مەن مەكتەپكە بارىمەن.")
for w in doc.words:
    print(f"{w.text:<20} lemma={w.lemma:<15} feats={w.feats}")

## 5. POS Tagging, Lemmatisation, and Dependency Parsing (UDT treebank)

In [ ]:
nlp_parse = Pipeline(
    "uig",
    processors=["tokenize", "pos", "lemma", "depparse"],
    script="Arab",
)
doc = nlp_parse("ئۈرۈمچى شىنجاڭنىڭ پايتەختى.")
print(f"{'Word':<20} {'UPOS':<8} {'Lemma':<15} {'Deprel'}")
for w in doc.words:
    print(f"{w.text:<20} {w.upos:<8} {w.lemma:<15} {w.deprel}")
print("\nCoNLL-U:\n", doc.to_conllu())

## 6. Translation

In [ ]:
turkicnlp.download("uig", processors=["translate"])
trans = Pipeline("uig", processors=["translate"], translate_tgt_lang="eng_Latn")
doc = trans("شىنجاڭ ئۇيغۇر ئاپتونوم رايونى.")
print("EN:", doc.translation)